#  1. Предварительный анализ данных

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

train_df = pd.read_csv('/kaggle/input/21vek-query-classification/train.csv')
categories_df = pd.read_csv('/kaggle/input/21vek-query-classification/categories.csv')
test_df = pd.read_csv('/kaggle/input/21vek-query-classification/test.csv')

In [ ]:
train_df.shape, train_df.isnull().sum(), train_df.duplicated().sum()

In [ ]:
train_df = train_df.merge(categories_df, on="CategoryID", how="left")

Размер данных достаточно большой, значит примеров для обучения много. Пропущенных значений и дубликатов нет.

In [ ]:
train_df.head()

In [ ]:
plt.figure(figsize=(12,6))
top_categories = train_df["CategoryName"].value_counts().head(20)
sns.barplot(x=top_categories.values, y=top_categories.index, palette = "viridis")
plt.show()

train_df["query_length"] = train_df["Query"].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,5))
sns.histplot(train_df["query_length"], bins=30, kde=True, color="coral")
plt.grid(True)
plt.show()

* На первом графике видно, что есть категории с большим количеством запросов, а есть категории, где запросов очень мало. При таком раскладе модель будет хуже предсказывать редкие классы. Это я исправлю путем использования взвешенных моделей.
* На гистограмме видно, что в основном длина запроса короткая, поэтому для обучения моделей использую векторайзер tf-idf.

In [ ]:
category_statistics = train_df["CategoryName"].value_counts()
print(round(category_statistics.mean(),2), category_statistics.median(),"\n", category_statistics.min(), category_statistics.max())

Среднее кол-во примеров на категорию 175. Медиана отличается от среднего, поэтому распределение категорий неравномерное. Максимум и минимум сильно различаются. Все это говорит о дисбалансе классов, поэтому надо подобрать такие модели, которые эту проблему решают наиболее эффективно.

# 2. Обучение моделей

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier 
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score 
import re
import pandas as pd

Немного преобразую текст в столбце с запросами, чтобы модель обучалась лучше.

In [ ]:
def cleaning(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zа-я0-9]+", " ", text)
    return text.strip()

train_df["clean_query"] = train_df["Query"].apply(cleaning)
train_df.head()

In [ ]:
X = train_df["clean_query"]
Y = train_df["CategoryID"]

X_train, X_value, y_train, y_value = train_test_split(X, Y, test_size = 0.2, stratify = Y, random_state = 42)

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range = (1,2))
X_train_vect = vectorizer.fit_transform(X_train)
X_value_vect = vectorizer.transform(X_value)

Я попробую обучить 3 модели, которые лучше всего подходят для работы с текстом, и затем выберу ту модель, которая покажет самую высокую точность. 

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, solver='saga'),
    "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=1000, random_state=42)
}

In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train_vect, y_train)
    preds = model.predict(X_value_vect)
    acc = accuracy_score(y_value, preds) 
    results[name] = (acc, model)
    print(f"{name} Accuracy: {acc:.4f}")

In [ ]:
best_model_name = max(results, key=lambda x: results[x][0]) 
best_model = results[best_model_name][1]
print(f"\n Лучшая модель: {best_model_name}")

Самую высокую точность показала модель RandomForestClassifier, поэтому для прогнозирования на тестовых данных буду использовать именно её.

# 3. Предсказание на тестовых данных

In [ ]:
test_df["clean_query"] = test_df["Query"].apply(cleaning)
X_test_vect = vectorizer.transform(test_df["clean_query"])
test_preds = best_model.predict(X_test_vect)

In [ ]:
result_df = pd.DataFrame({
    "ID": test_df["ID"], 
    "CategoryID": test_preds      
})
result_df.head()

In [ ]:
result_df.to_csv("submission.csv", index=False)